# E2 (label delay) on Kaggle: Logistic, Random Forest, XGBoost

**Setup:** Settings → Accelerator **None** (no GPU needed), Internet **On**. No input file and no secret are needed.

**Run:** click **Save Version → Save & Run All (Commit)**. About 3 to 4 hours.
When done, download from the version's **Output**: `e2_rows.csv`, `e2_table.csv`, `e2_summary.csv`,
`versions.json` and the two charts.

Results are saved after every model and window. If a run is interrupted, upload the partial `e2_rows.csv`
as an input dataset and run again: finished models and windows are skipped.

## 1. Install, environment, earlier results

In [ ]:
# 1. Install (pinned to the E0 versions), environment, earlier results
!pip install -q scikit-learn==1.6.1 xgboost==3.4.1
import os, sys, glob, json, shutil, importlib.metadata as md
import pandas as pd

versions = {"python": sys.version.split()[0]}
for p in ["numpy", "pandas", "scikit-learn", "xgboost"]:
    try: versions[p] = md.version(p)
    except md.PackageNotFoundError: versions[p] = "not installed"
print(versions)
json.dump(versions, open("versions.json", "w"), indent=1)

# Earlier results (input dataset), so finished models and windows are skipped
if not os.path.exists("e2_rows.csv"):
    found = glob.glob("/kaggle/input/**/e2_rows*.csv", recursive=True)
    if found:
        best = max(found, key=lambda f: len(pd.read_csv(f)))
        shutil.copy(best, "e2_rows.csv"); print("copied", best)
if os.path.exists("e2_rows.csv"):
    d = pd.read_csv("e2_rows.csv"); print("existing rows:", len(d)); print(d.groupby("model").window.unique())
else:
    print("No earlier e2_rows.csv: starting fresh.")

## 2. Data and models

In [ ]:
# 2. Data and models (identical configuration to E0 and E1)
import numpy as np, urllib.request
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

if not Path("elliptic_labelled.parquet").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
                               "main/resources/elliptic/elliptic_labelled_v2.parquet", "elliptic_labelled.parquet")
ell = pd.read_parquet("elliptic_labelled.parquet")
feat_cols = [c for c in ell.columns if c.startswith("feat_")]
X = ell[feat_cols].values
y = (ell["class"] == 1).astype(int).values
ts = ell["time_step"].values
boundaries = [1, 10, 20, 30, 40, 50]
assert len(y) == 46564 and y.sum() == 4545
print("Data check passed")

def make_model(name, seed=42):
    if name == "Logistic":
        return Pipeline([("sc", StandardScaler()),
                         ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced",
                                      random_state=seed, n_jobs=-1)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                             random_state=seed, eval_metric="logloss", n_jobs=-1)
    raise ValueError(name)

def fit_proba(name, tr_idx, te_mask, seed=42):
    m = make_model(name, seed)
    ytr = y[tr_idx]
    if name == "XGBoost":   # class weight from THIS training set only (no look-ahead)
        m.set_params(scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum())
    m.fit(X[tr_idx], ytr)
    Xte = X[te_mask]
    return np.concatenate([m.predict_proba(Xte[k:k + 5000])[:, 1] for k in range(0, len(Xte), 5000)])

## 3. Settings

In [ ]:
# 3. Settings
MODELS = ["Logistic", "Random Forest", "XGBoost"]
E2_R = {"Logistic": 50, "Random Forest": 50, "XGBoost": 50}   # control-arm repeats
KS = [1, 2, 4, 6]                      # label delay in periods (about 2 weeks each)
SEEDS = [42, 1, 7, 21, 99]             # model seeds for the full and delay arms

## 4. E2: label delay

In [ ]:
# 4. E2: label delay with a random-removal control arm, 4 walk-forward windows
#   delay arm  : drop the k most recent training periods
#   control arm: drop the SAME number of transactions per class, at random from the whole past
#   data loss    = AUC(no delay) - AUC(control);  recency loss = AUC(control) - AUC(delay)
import time
from sklearn.metrics import roc_auc_score, average_precision_score
OUT = Path("e2_rows.csv")
done = set()
if OUT.exists():
    d = pd.read_csv(OUT); done = set(zip(d.model, d.window))

all_idx = np.arange(len(y))
for name in MODELS:
    stochastic = name != "Logistic"
    seeds = SEEDS if stochastic else [42]
    for i in range(len(boundaries) - 2):
        cut, hi = boundaries[i + 1], boundaries[i + 2]
        win = f"t={cut}-{hi-1}"
        if (name, win) in done:
            continue
        t0 = time.time()
        test = (ts >= cut) & (ts < hi)
        past = ts < cut
        rows = []
        def score(arm, k, rep, seed, tr_idx):
            p = fit_proba(name, tr_idx, test, seed)
            rows.append({"model": name, "window": win, "arm": arm, "k": k, "rep": rep, "seed": seed,
                         "train_n": len(tr_idx), "train_illicit": int(y[tr_idx].sum()),
                         "auc": roc_auc_score(y[test], p), "ap": average_precision_score(y[test], p)})
        for s in seeds:
            score("full", 0, 0, s, all_idx[past])
        for k in KS:
            if cut - k < 5:            # too little history (t=10-19 with k=6 leaves only 3 periods)
                continue
            for s in seeds:
                score("delay", k, 0, s, all_idx[past & (ts < cut - k)])
            lost = past & (ts >= cut - k)
            for r in range(E2_R[name]):
                rng = np.random.default_rng(r)
                keep = past.copy()
                for lab in (0, 1):
                    pool = np.where(past & (y == lab))[0]
                    n = int((lost & (y == lab)).sum())
                    keep[rng.choice(pool, n, replace=False)] = False
                score("control", k, r, r if stochastic else 42, all_idx[keep])
        pd.DataFrame(rows).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
        print(f"{name:13s} {win} done ({time.time() - t0:.0f}s)", flush=True)
print("E2 done", flush=True)
d = pd.read_csv(OUT); print("final rows:", len(d)); print(d.groupby("model").window.unique())

## 5. Tables and charts

In [ ]:
# 5. Tables and charts
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
ORDER = ["Logistic", "Random Forest", "XGBoost"]
INK, MUTED, GRID, AXIS = "#0b0b0b", "#6b6a63", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": AXIS, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 150})
pct = lambda s, q: np.percentile(s, q)
# ---------------- E2 ----------------
if Path("e2_rows.csv").exists():
    e2 = pd.read_csv("e2_rows.csv")
    models = [m for m in ORDER if m in e2.model.unique()]
    rows = []
    for (m, w), g in e2.groupby(["model", "window"]):
        full = g[g.arm == "full"]
        a_full, p_full = full.auc.mean(), full.ap.mean()
        rows.append({"Model": m, "Window": w, "k": 0, "Train n": int(full.train_n.iloc[0]),
                     "AUC delay": a_full, "AUC control": a_full, "AUC control 95% CI": "",
                     "Data loss": 0.0, "Recency loss": 0.0, "Total loss": 0.0,
                     "AP delay": p_full, "AP control": p_full, "AP data loss": 0.0, "AP recency loss": 0.0})
        for k in sorted(g[g.arm == "delay"].k.unique()):
            dl = g[(g.arm == "delay") & (g.k == k)]
            ct = g[(g.arm == "control") & (g.k == k)]
            rows.append({"Model": m, "Window": w, "k": int(k), "Train n": int(dl.train_n.iloc[0]),
                         "AUC delay": dl.auc.mean(), "AUC control": ct.auc.mean(),
                         "AUC control 95% CI": f"[{pct(ct.auc, 2.5):.4f}, {pct(ct.auc, 97.5):.4f}]",
                         "Data loss": a_full - ct.auc.mean(), "Recency loss": ct.auc.mean() - dl.auc.mean(),
                         "Total loss": a_full - dl.auc.mean(),
                         "AP delay": dl.ap.mean(), "AP control": ct.ap.mean(),
                         "AP data loss": p_full - ct.ap.mean(), "AP recency loss": ct.ap.mean() - dl.ap.mean()})
    t2 = pd.DataFrame(rows)
    t2["Model"] = pd.Categorical(t2.Model, [m for m in ORDER if m in models], ordered=True)
    t2["wnum"] = t2.Window.str.extract(r"=(\d+)").astype(int)
    t2 = t2.sort_values(["Model", "wnum", "k"]).drop(columns="wnum")
    t2.round(4).to_csv("e2_table.csv", index=False)
    print("\nE2: label delay, delay arm vs random-removal control (AUC; losses positive = worse)")
    print(t2[["Model", "Window", "k", "Train n", "AUC delay", "AUC control", "Data loss",
              "Recency loss", "Total loss"]].round(4).to_string(index=False))

    # summary over windows t=20-49, where every delay level exists
    sub = t2[(t2.k > 0) & (t2.Window != "t=10-19")]
    summ = sub.groupby(["Model", "k"], observed=True)[["Data loss", "Recency loss", "Total loss",
                                                       "AP data loss", "AP recency loss"]].mean()
    summ.round(4).to_csv("e2_summary.csv")
    print("\nE2 summary, mean over windows t=20-49:")
    print(summ.round(4).to_string())

    # Chart 1: AUC vs delay, one line per window, small multiples by model (delay solid, control dashed)
    wins = sorted(t2.Window.unique(), key=lambda s: int(s.split("=")[1].split("-")[0]))
    WCOL = dict(zip(wins, ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]))
    fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 3.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, models):
        for w in wins:
            g = t2[(t2.Model == m) & (t2.Window == w)]
            ax.plot(g.k, g["AUC delay"], "-o", color=WCOL[w], lw=2, ms=5, label=w)
            ax.plot(g.k, g["AUC control"], "--", color=WCOL[w], lw=1.2, alpha=0.8)
        ax.set_title(m, fontsize=10, color=INK); ax.set_xlabel("Label delay k (periods, 2 weeks each)")
        ax.set_xticks([0, 1, 2, 4, 6]); ax.grid(color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    axes[0].set_ylabel("Walk-forward AUC"); axes[0].legend(frameon=False, fontsize=8, title="Test window",
                                                           title_fontsize=8)
    fig.suptitle("E2. AUC by label delay. Solid: delay arm. Dashed: random-removal control.",
                 fontsize=11, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.94)); fig.savefig("e2_chart_lines.png"); plt.close(fig)

    # Chart 2: decomposition (stacked; negative segments stack below zero), mean over t=20-49
    fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 3.6), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, models):
        s = summ.loc[m]
        ks = [str(k) for k in s.index]
        dl, rc = s["Data loss"].values, s["Recency loss"].values
        pos_b, neg_b = np.zeros(len(ks)), np.zeros(len(ks))
        for vals, c, lab in [(dl, "#2a78d6", "Data loss"), (rc, "#eb6834", "Recency loss")]:
            bottom = np.where(vals >= 0, pos_b, neg_b)
            ax.bar(ks, vals, 0.6, bottom=bottom, color=c, label=lab, edgecolor="white", linewidth=1.5)
            pos_b += np.clip(vals, 0, None); neg_b += np.clip(vals, None, 0)
        ax.plot(ks, s["Total loss"].values, "D", color=INK, ms=4, label="Total")
        ax.axhline(0, color=AXIS, lw=1); ax.set_title(m, fontsize=10, color=INK)
        ax.set_xlabel("Label delay k (periods)"); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
    axes[0].set_ylabel("AUC lost vs no delay"); axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle("E2. Decomposition of AUC loss, mean over test windows t=20-49", fontsize=11, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.93)); fig.savefig("e2_chart_decomposition.png"); plt.close(fig)
print("\nanalysis done")

## 6. Charts

In [ ]:
# 6. Show charts
from IPython.display import Image, display
for f in ["e2_chart_lines.png", "e2_chart_decomposition.png"]:
    display(Image(f))